# Downloader

This small script will help us download the images from the Azure Data Lake

## Install Dependencies

In [11]:
#r "nuget: Azure.Identity, 1.17.1"
#r "nuget: Azure.Storage.Blobs, 12.27.0-beta.1"
#r "nuget: CsvHelper, 33.1.0"

Installed Packages Azure.Identity, 1.17.1 Azure.Storage.Blobs, 12.27.0-beta.1 CsvHelper, 33.1.0

## Add Usings

In [12]:
using System;
using System.Globalization;
using System.IO;
using Azure.Identity;
using Azure.Storage.Blobs;
using CsvHelper;
using CsvHelper.Configuration;

## Generate the CSV

To do it, we will execute the following SQL:
```sql
SELECT
    [Id],
    [Matricula],
    [LinkImagenVehiculo1],
    [LinkImagenVehiculo2]
FROM
    [HistoricoTransitoVehiculos]
```

# Parse the CSV

In [3]:
public record HistoricoTransitoVehiculos
{
    public long Id { get; init; }
    public string Matricula { get; set; }
    public string LinkImagenVehiculo1 { get; set; }
    public string LinkImagenVehiculo2 { get; set; }
}

var fileInfo = new FileInfo("historico-transito-vehiculos.csv");
var fileStream = fileInfo.Open(FileMode.Open, FileAccess.Read, FileShare.Read);
var streamReader = new StreamReader(fileStream);
var csvReader = new CsvReader(streamReader, new CsvConfiguration(CultureInfo.InvariantCulture) { HasHeaderRecord = true, Delimiter = ","});
var records = csvReader.GetRecords<HistoricoTransitoVehiculos>();

## Connect to Azure

In [14]:
var accountUri = new Uri("https://stgghlmtransauto01.blob.core.windows.net/");
var blobServiceClient = new BlobServiceClient(accountUri, new DefaultAzureCredential());
var blobContainerClient = blobServiceClient.GetBlobContainerClient("hmeta");

## Download Image

In [24]:

var directoryInfo = new DirectoryInfo("./images/");
directoryInfo.Create();

foreach (var historicoTransitoVehiculo in records.Take(10)) {
    var idDirectory = new DirectoryInfo($"{directoryInfo.FullName}{historicoTransitoVehiculo.Id}");
    idDirectory.Create();

    try{
        var fileInfo = new FileInfo($"{idDirectory.FullName}/{historicoTransitoVehiculo.Matricula}-link1.jpeg");
        if(!fileInfo.Exists) {
            var link = new Uri(historicoTransitoVehiculo.LinkImagenVehiculo1);
            var blobClient = blobContainerClient.GetBlobClient(link.AbsolutePath);
            blobClient.DownloadTo(fileInfo.FullName);
        }
    }catch(Exception){}

    try{
        var fileInfo = new FileInfo($"{idDirectory.FullName}/{historicoTransitoVehiculo.Matricula}-link2.jpeg");
        if(!fileInfo.Exists) {
            var link = new Uri(historicoTransitoVehiculo.LinkImagenVehiculo2);
            var blobClient = blobContainerClient.GetBlobClient(link.AbsolutePath);
            blobClient.DownloadTo(fileInfo.FullName);
        }
    }catch(Exception){}
}